In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import us

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Analyzing and Predicting Housing Prices Using Socioeconomic, Crime and Environmental Factors in USA

## 1. Introduction

Housing prices are influenced by a wide range of factors including economic conditions, demographic characteristics, public safety and environmental quality. While traditional real estate analysis often focuses primarily on property characteristics and income-related indicators, broader socio-economic and environmental factors may also contribute significantly to housing valuation.

This project aims to analyze and predict housing prices in the United States by integrating multiple data sources, iincluding housing data, crime statistics and air quality measurments. By combining these datasets the project provides a more comprehensive view of the factors that influence property values.

In addition to traditional housing features such as income, population, and property characteristics, this study incorporates crime-related variables and air pollution indicators to evaluate their impact on real estate prices.

Using data preprocessing, exploratory analysis and machine learning techniques, the project investigates both the relationships among these variables and their predictive power. The objective is not only to identify key factors affecting housing prices but also to build a model capable of accurately predicting property values based on a diverse set of socio-economic and environmental features. 

## 2. Data Overview

This project combines multiple datasets to analyze housing prices in relation to environmental and socioeconomic factors.

The main datasets used include:
* A housing dataset containg property-related features such as income, number of rooms, house age, price and population. => https://www.kaggle.com/datasets/vedavyasv/usa-housing?resource=download
* A crime dataset providing information aboout different types of crimes, including violent crime and property crime. => https://www.kaggle.com/datasets/kabhishm/united-states-crime-rates-by-city-population?select=crime_60_100.csv
* An air quality dataset containing poluution indicators such as NO2, O3, SO2 and CO levels. => https://www.kaggle.com/datasets/sogun3/uspollution

These dataset were merged based on geographical information (state-level data), allowing the integration of multiple independent data sources into a unified dataset.

The final dataset includes both numerical and categorical variables and represents multiple aspects of the living environment.

# 3. Data Processing

### 3.1 Housing Data

In [2]:
housing_data = pd.read_csv("data/USA_Housing.csv")

In [3]:
housing_data.head(10)

,Avg. Area Income,Avg. Area House Age,Avg. Area Number of Rooms,Avg. Area Number of Bedrooms,Area Population,Price,Address
0,79545.458574,5.682861,7.009188,4.09,23086.800503,1.059034e+06,"208 Michael Ferry Apt. 674\nLaurabury, NE 3701..."
1,79248.642455,6.002900,6.730821,3.09,40173.072174,1.505891e+06,"188 Johnson Views Suite 079\nLake Kathleen, CA..."
2,61287.067179,5.865890,8.512727,5.13,36882.159400,1.058988e+06,"9127 Elizabeth Stravenue\nDanieltown, WI 06482..."
3,63345.240046,7.188236,5.586729,3.26,34310.242831,1.260617e+06,USS Barnett\nFPO AP 44820
4,59982.197226,5.040555,7.839388,4.23,26354.109472,6.309435e+05,USNS Raymond\nFPO AE 09386
5,80175.754159,4.988408,6.104512,4.04,26748.428425,1.068138e+06,"06039 Jennifer Islands Apt. 443\nTracyport, KS..."
6,64698.463428,6.025336,8.147760,3.41,60828.249085,1.502056e+06,"4759 Daniel Shoals Suite 442\nNguyenburgh, CO ..."
7,78394.339278,6.989780,6.620478,2.42,36516.358972,1.573937e+06,"972 Joyce Viaduct\nLake William, TN 17778-6483"
8,59927.660813,5.362126,6.393121,2.30,29387.396003,7.988695e+05,USS Gilbert\nFPO AA 20957
9,81885.927184,4.423672,8.167688,6.10,40149.965749,1.545155e+06,Unit 9446 Box 0958\nDPO AE 97025


In [4]:
housing_data.shape

(5000, 7)

In [5]:
housing_data.isnull().sum()

Avg. Area Income                0
Avg. Area House Age             0
Avg. Area Number of Rooms       0
Avg. Area Number of Bedrooms    0
Area Population                 0
Price                           0
Address                         0
dtype: int64

In [6]:
housing_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Avg. Area Income              5000 non-null   float64
 1   Avg. Area House Age           5000 non-null   float64
 2   Avg. Area Number of Rooms     5000 non-null   float64
 3   Avg. Area Number of Bedrooms  5000 non-null   float64
 4   Area Population               5000 non-null   float64
 5   Price                         5000 non-null   float64
 6   Address                       5000 non-null   str    
dtypes: float64(6), str(1)
memory usage: 273.6 KB


In [7]:
housing_data.describe().T

,count,mean,std,min,25%,50%,75%,max
Avg. Area Income,5000.0,6.858311e+04,10657.991214,17796.631190,61480.562388,6.880429e+04,7.578334e+04,1.077017e+05
Avg. Area House Age,5000.0,5.977222e+00,0.991456,2.644304,5.322283,5.970429e+00,6.650808e+00,9.519088e+00
Avg. Area Number of Rooms,5000.0,6.987792e+00,1.005833,3.236194,6.299250,7.002902e+00,7.665871e+00,1.075959e+01
Avg. Area Number of Bedrooms,5000.0,3.981330e+00,1.234137,2.000000,3.140000,4.050000e+00,4.490000e+00,6.500000e+00
Area Population,5000.0,3.616352e+04,9925.650114,172.610686,29403.928702,3.619941e+04,4.286129e+04,6.962171e+04
Price,5000.0,1.232073e+06,353117.626581,15938.657923,997577.135049,1.232669e+06,1.471210e+06,2.469066e+06


The housing dataset contains 5000 observations and 7 variables providing key information about residentail properties.

The dataset includes several numerical features such as average area income, house age, number of rooms, number of bedrooms and population, as well as the target variable - housing price. Additionally, it contains and address column which represents the location of each property but is not directly used in the modeling process.

All numerical variables are complete, with no missing values, making the dataset suitable for direct analysis. This features are represented as floating-point numbers, while address columns is stored as string.

Summary statistics show that the average area income is approximately 68 000 while the average house price is around 1.23 million. The number of rooms average close to 7, indicating relatively large residential properties.

The dataset provides a solid foundation for analyzing the relationship between housing characteristics and property prices before integrating additional external factors such as crime and air quality.

In [8]:
housing_data["State"] = housing_data["Address"].str.extract(r'\b([A-Z]{2})\s+\d{5}\b')
housing_data = housing_data[housing_data.State.str.isupper()]

As part of the data processing stage, geographical information was extracted from the address field in the sousing dataset. SInce the original dataset did not include a separate state columns, a regular expression was included to identify and extract the state abbreviation from the address string. Specifically, the state code was extracted by identifying patterns consisting of two uppercase letters followed by ZIP code. This allowed the creation of a new **State** feature, which was later used to merge the housing dataset with external datasets containing crime statistics and air quality data. 

In [9]:
valid_states = [state.abbr for state in us.states.STATES]
housing_data = housing_data[housing_data.State.isin(valid_states)]
housing_data.isnull().sum()

Avg. Area Income                0
Avg. Area House Age             0
Avg. Area Number of Rooms       0
Avg. Area Number of Bedrooms    0
Area Population                 0
Price                           0
Address                         0
State                           0
dtype: int64

To ensure data quality only valid state abbreviations were retained. This step was essential for aligning the datasets and enabling accurate integration based on geographical information.

### 3.2 Crime Data

In [10]:
crime_data_40_60 = pd.read_csv("data/crime_40_60.csv")
crime_data_40_60.head(5)

,states,cities,population,violent_crime,murder,rape,robbery,agrv_assault,prop_crime,burglary,larceny,vehicle_theft
0,Pennsylvania,"Abington Township, Montgomery County","55,731",197.4,1.8,14.4,70.0,111.2,1979.1,296.1,1650.8,32.3
1,Oregon,Albany,"51,084",86.1,0.0,19.6,45.0,21.5,3092.9,438.5,2470.4,184.0
2,Louisiana,Alexandria,"48,449",1682.2,18.6,28.9,293.1,1341.6,7492.4,2010.4,5102.3,379.8
3,California,Aliso Viejo,"48,999",87.8,0.0,0.0,12.2,75.5,847.0,208.2,612.3,26.5
4,Florida,Altamonte Springs,"42,296",335.7,2.4,21.3,82.8,229.3,3057.0,427.9,2463.6,165.5


In [11]:
crime_data_40_60.shape

(358, 12)

In [12]:
crime_data_40_60.info()

<class 'pandas.DataFrame'>
RangeIndex: 358 entries, 0 to 357
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   states         358 non-null    str    
 1   cities         358 non-null    str    
 2   population     358 non-null    str    
 3   violent_crime  350 non-null    float64
 4   murder         358 non-null    float64
 5   rape           352 non-null    float64
 6   robbery        358 non-null    float64
 7   agrv_assault   356 non-null    float64
 8   prop_crime     357 non-null    float64
 9   burglary       357 non-null    float64
 10  larceny        358 non-null    float64
 11  vehicle_theft  358 non-null    float64
dtypes: float64(9), str(3)
memory usage: 33.7 KB


In [13]:
crime_data_40_60.describe().T

,count,mean,std,min,25%,50%,75%,max
violent_crime,350.0,308.175714,286.503643,12.0,121.700,223.05,394.350,2362.1
murder,358.0,3.017598,5.081112,0.0,0.000,1.80,4.075,52.7
rape,352.0,25.881250,23.664427,0.0,8.450,19.05,37.200,178.7
robbery,358.0,84.647765,91.400726,0.0,26.225,54.60,112.850,779.9
agrv_assault,356.0,190.373315,204.143095,4.5,60.750,129.35,236.075,1856.9
prop_crime,357.0,2859.556863,1402.755755,650.5,1816.400,2592.00,3521.400,8225.2
burglary,357.0,606.110364,425.179055,69.9,304.700,478.00,779.200,2368.9
larceny,358.0,2074.473184,1014.901448,468.6,1331.625,1839.90,2608.250,5764.6
vehicle_theft,358.0,178.441341,160.102237,1.7,67.825,130.00,234.625,1118.4


In [14]:
crime_data_40_60.isnull().sum()

states           0
cities           0
population       0
violent_crime    8
murder           0
rape             6
robbery          0
agrv_assault     2
prop_crime       1
burglary         1
larceny          0
vehicle_theft    0
dtype: int64

The first crime dataset contains information about cities with population between 40 000 and 60 000 residents. It includes 358 observations and 12 variables, covering various types of crime such as violient crime, murder, rape, robbery, aggravated assault and property crime. Most variables are numeric and represent crime rates per population, allowing for meaningful comparisons across cities. A small number of missing values were identified in some columns. The dataset provides a detailed view of crime patterns in medium-sized cities and serves as an important components for analyzing how crime levels influence housing prices. 

In [15]:
crime_data_60_100 = pd.read_csv("data/crime_60_100.csv")
crime_data_60_100.head(5)

,states,cities,population,violent_crime,murder,rape,robbery,agrv_assault,prop_crime,burglary,larceny,vehicle_theft
0,California,Alameda,"75,467",212.0,1.3,11.91,106.0,92.8,"2,507.1",392.2,"1,723.9",390.9
1,Georgia,Albany,"78,512","1,035.5",5.1,34.4,285.3,710.7,"6,369.7","1,793.4","4,291.1",285.3
2,New York,Albany,"98,187",816.8,4.1,43.8,253.6,515.3,"4,420.1",903.4,"3,359.9",156.8
3,California,Alhambra,"84,469",176.4,-,2.4,78.1,95.9,"2,271.8",384.8,"1,585.2",301.9
4,Texas,Allen,"88,783",61.9,-,12.4,14.6,34.9,"1,612.9",242.2,"1,321.2",49.6


In [16]:
crime_data_60_100.shape

(305, 12)

In [17]:
crime_data_60_100.info()

<class 'pandas.DataFrame'>
RangeIndex: 305 entries, 0 to 304
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   states         305 non-null    str    
 1   cities         305 non-null    str    
 2   population     305 non-null    str    
 3   violent_crime  295 non-null    str    
 4   murder         305 non-null    str    
 5   rape           295 non-null    str    
 6   robbery        305 non-null    float64
 7   agrv_assault   305 non-null    str    
 8   prop_crime     304 non-null    str    
 9   burglary       304 non-null    str    
 10  larceny        305 non-null    str    
 11  vehicle_theft  305 non-null    str    
dtypes: float64(1), str(11)
memory usage: 28.7 KB


In [18]:
crime_data_60_100.describe().T

,count,mean,std,min,25%,50%,75%,max
robbery,305.0,116.519344,116.787983,4.8,38.9,81.6,152.3,972.1


In [19]:
crime_data_60_100.isnull().sum()

states            0
cities            0
population        0
violent_crime    10
murder            0
rape             10
robbery           0
agrv_assault      0
prop_crime        1
burglary          1
larceny           0
vehicle_theft     0
dtype: int64

The second dataset focuses on cities with populations between 60 000 and 100 000 residents and contains 305 observations. Similar to the first dataset, it includes multiple crime-related variables such as violent crime, robbery and assult. Some columns were initially stored as strings instead of numerical values, requiring type conversion during processing. This dataset expands the analysis by incorporating larger urban areas and contributes to a more comprehensive understanding of crime distribution.

In [20]:
crime_data_100_250 = pd.read_csv("data/crime_100_250.csv")
crime_data_100_250.head(5)

,states,cities,population,violent_crime,murder,rape,robbery,agrv_assault,prop_crime,burglary,larceny,vehicle_theft
0,Texas,Abilene,"119,886",393.7,2.5,31.7,105.9,253.6,"3,664.3",865,"2,656.7",142.6
1,Ohio,Akron,"198,390",886.6,12.1,84.2,290.8,499.5,"5,057.7","1,728.4","2,965.9",363.4
2,Virginia,Alexandria,"145,892",166.6,-,6.2,94.6,65.8,"2,049.5",192.6,"1,633.4",223.5
3,Pennsylvania,Allentown,"119,334",547.2,12.6,45.3,313.4,176,"3,857.2","1,045.8","2,503.1",308.4
4,Texas,Amarillo,"196,576",650.1,5.1,56.0,141.4,447.7,"4,527.5","1,061.7","3,145.9",320


In [21]:
crime_data_100_250.shape

(212, 12)

In [22]:
crime_data_100_250.info()

<class 'pandas.DataFrame'>
RangeIndex: 212 entries, 0 to 211
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   states         212 non-null    str    
 1   cities         212 non-null    str    
 2   population     212 non-null    str    
 3   violent_crime  211 non-null    str    
 4   murder         212 non-null    str    
 5   rape           211 non-null    float64
 6   robbery        212 non-null    float64
 7   agrv_assault   212 non-null    str    
 8   prop_crime     212 non-null    str    
 9   burglary       212 non-null    str    
 10  larceny        212 non-null    str    
 11  vehicle_theft  212 non-null    str    
dtypes: float64(2), str(10)
memory usage: 20.0 KB


In [23]:
crime_data_100_250.describe().T

,count,mean,std,min,25%,50%,75%,max
rape,211.0,32.128436,25.697522,2.2,15.8,27.6,41.550,265.7
robbery,212.0,157.465094,124.355664,13.3,62.0,129.2,212.375,662.2


In [24]:
crime_data_100_250.isnull().sum()

states           0
cities           0
population       0
violent_crime    1
murder           0
rape             1
robbery          0
agrv_assault     0
prop_crime       0
burglary         0
larceny          0
vehicle_theft    0
dtype: int64

The third dataset contains crime data for cities with populations between 100 000 and 250 000 residents with total of 212 observations. The structure is consistent with previous datasets including similar crime indicators. The dataset requires preprocessing steps such as converting data types and handling a small number of missing values. Compared to smaller cities, crime rates in this dataset show greater variability, reflecting the complexity of larger urban environments. This dataset plays a key role in capturing crime dynamics in larger metropolitan areas,

In [25]:
crime_data_250_plus = pd.read_csv("data/crime_250_plus.csv")
crime_data_250_plus.head(5)

,states,cities,population,total_crime,murder,rape,robbery,agrv_assault,tot_violent_crime,burglary,larceny,vehicle_theft,tot_prop_crim,arson
0,Alabama,Mobile3,"248,431",6217.02,20.13,58.16,177.11,485.85,740.25,"1,216.84","3,730.21",506.78,"5,453.83",22.94
1,Alaska,Anchorage,"296,188",6640.04,9.12,132.01,262.67,799.49,"1,203.29",748.17,"3,619.66","1,047.98","5,415.82",20.93
2,Arizona,Chandler,"249,355",2589.08,2.01,52.13,56.95,148.38,259.47,314.41,"1,866.01",149.18,"2,329.61",NaN
3,Arizona,Gilbert,"242,090",1483.75,2.07,16.11,21.07,46.26,85.51,192.49,"1,137.59",55.76,"1,385.85",12.39
4,Arizona,Glendale,"249,273",5037.85,4.81,38.91,192.96,251.53,488.22,637.45,"3,426.36",466.56,"4,530.37",19.26


In [26]:
crime_data_250_plus = crime_data_250_plus.rename(columns={
    "tot_violent_crime" : "violent_crime",
    "tot_prop_crim" : "prop_crime"
})

In [27]:
crime_data_250_plus.shape

(100, 14)

In [28]:
crime_data_250_plus.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   states         100 non-null    str    
 1   cities         100 non-null    str    
 2   population     100 non-null    str    
 3   total_crime    100 non-null    float64
 4   murder         100 non-null    float64
 5   rape           96 non-null     float64
 6   robbery        100 non-null    float64
 7   agrv_assault   100 non-null    str    
 8   violent_crime  96 non-null     str    
 9   burglary       100 non-null    str    
 10  larceny        100 non-null    str    
 11  vehicle_theft  100 non-null    str    
 12  prop_crime     100 non-null    str    
 13  arson          92 non-null     float64
dtypes: float64(5), str(9)
memory usage: 11.1 KB


In [29]:
crime_data_250_plus.describe().T

,count,mean,std,min,25%,50%,75%,max
total_crime,100.0,4312.471900,1674.275944,1381.31,3029.7800,4091.500,5392.1750,8734.98
murder,100.0,11.618400,10.940627,0.72,4.1850,8.595,15.7025,66.07
rape,96.0,59.942083,28.818067,13.85,38.4650,56.570,75.4425,144.67
robbery,100.0,229.274100,159.584269,19.92,127.5400,191.090,306.5125,958.71
arson,92.0,24.661957,20.870553,0.73,10.2775,18.370,33.3600,129.55


In [30]:
crime_data_250_plus.isnull().sum()

states           0
cities           0
population       0
total_crime      0
murder           0
rape             4
robbery          0
agrv_assault     0
violent_crime    4
burglary         0
larceny          0
vehicle_theft    0
prop_crime       0
arson            8
dtype: int64

The final dataset includes cities with populations above 250 000 and contains 100 observations. Some inconsistence in columnsnaming were identified and corrected to ensure alignment with other datasets. This dataset represents large metropolitan areas and provides insight into crime patterns in highly populated regions.

#### Combine Crime Dataset

In [31]:
crime_all_data = pd.concat([crime_data_40_60, crime_data_60_100, crime_data_100_250, crime_data_250_plus], ignore_index=True)

In [32]:
crime_all_data.shape

(975, 14)

In [33]:
crime_all_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 975 entries, 0 to 974
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   states         975 non-null    str    
 1   cities         975 non-null    str    
 2   population     975 non-null    str    
 3   violent_crime  952 non-null    object 
 4   murder         975 non-null    object 
 5   rape           954 non-null    object 
 6   robbery        975 non-null    float64
 7   agrv_assault   973 non-null    object 
 8   prop_crime     973 non-null    object 
 9   burglary       973 non-null    object 
 10  larceny        975 non-null    object 
 11  vehicle_theft  975 non-null    object 
 12  total_crime    100 non-null    float64
 13  arson          92 non-null     float64
dtypes: float64(3), object(8), str(3)
memory usage: 106.8+ KB


In [34]:
crime_all_data.describe().T

,count,mean,std,min,25%,50%,75%,max
robbery,975.0,125.284421,123.387181,0.00,40.5500,84.00,167.865,972.10
total_crime,100.0,4312.471900,1674.275944,1381.31,3029.7800,4091.50,5392.175,8734.98
arson,92.0,24.661957,20.870553,0.73,10.2775,18.37,33.360,129.55


In [35]:
crime_all_data.isnull().sum()

states             0
cities             0
population         0
violent_crime     23
murder             0
rape              21
robbery            0
agrv_assault       2
prop_crime         2
burglary           2
larceny            0
vehicle_theft      0
total_crime      875
arson            883
dtype: int64

To create a unified dateset, the four crime datasets were combined into a single dataset representing cities of varying population sizes. The resulting combined dataset provides a comprehensive overview of crime across different urban environments, enabling more robust analysis when merged with housing and air quality data. This integration allows for the investigation of how crime rates, environmental conditions and population characteristics jointly influence housing prices.

Convert to correct data types:

In [36]:
numeric_cols = ["population", "violent_crime", "murder", "rape", "robbery", "agrv_assault", "prop_crime", "burglary", "larceny", "vehicle_theft", "total_crime"]

In [37]:
def clean_col(col):
    return pd.to_numeric(col.astype(str).str.replace(",", "").replace("-",""), errors="coerce")

In [38]:
for col in numeric_cols:
    if col in crime_all_data.columns:
        crime_all_data[col]=clean_col(crime_all_data[col])

In [39]:
crime_all_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 975 entries, 0 to 974
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   states         975 non-null    str    
 1   cities         975 non-null    str    
 2   population     975 non-null    int64  
 3   violent_crime  952 non-null    float64
 4   murder         864 non-null    float64
 5   rape           951 non-null    float64
 6   robbery        975 non-null    float64
 7   agrv_assault   973 non-null    float64
 8   prop_crime     973 non-null    float64
 9   burglary       973 non-null    float64
 10  larceny        975 non-null    float64
 11  vehicle_theft  975 non-null    float64
 12  total_crime    100 non-null    float64
 13  arson          92 non-null     float64
dtypes: float64(11), int64(1), str(2)
memory usage: 106.8 KB


A reusable preprocessing function was implemented to standarlize numerical columns across datasets. The function removes formatting inconsistencies such as commas and converts all relevant features into numeric types.

#### State Standartization

In [40]:
crime_all_data.states = crime_all_data.states.apply(
    lambda x: us.states.lookup(x).abbr if us.states.lookup(x) else None
)

In [41]:
crime_all_data[
    crime_all_data.states.str.len() !=2
]

,states,cities,population,violent_crime,murder,rape,robbery,agrv_assault,prop_crime,burglary,larceny,vehicle_theft,total_crime,arson
905,NaN,"Washington, D.C.",693972,948.74,16.72,63.84,338.77,529.42,4156.22,260.53,3528.96,366.73,5104.96,NaN


In [42]:
crime_all_data.loc[
    (crime_all_data.cities == "Washington, D.C."),
    "states"
] = "DC"

In [43]:
crime_all_data[
    crime_all_data.states.str.len() !=2
]

,states,cities,population,violent_crime,murder,rape,robbery,agrv_assault,prop_crime,burglary,larceny,vehicle_theft,total_crime,arson


As part of preprocessing stage, the state information in the crime dataset was srandardized to ensure consistency across all datasets. Since the original data contained full state names and in some cases non-standard values, the Python library **us** was used to convert all entries into official two-letter state abbreviations. The conversion was performed using lookup finction which maps each state name to its corresponding abbreviation. If a value could not be match to a valid U.S. state, it was replaced with a missing value(NaN).

After the transformation, additional validation was applied to ensure that all state entries consist of exactly two characters. Any remaining invalid or missing values were identified and corrected manually where possible. For example. entries such as "Washington, D.C." were handled separately to maintain consistency in the dataset. This step ensured that the state column is fully standardized and suitable for merging with other datasets, such as housing and air quality data, which alse rely on state-level identifiers.

### 3.3 Air Quality Data

In [44]:
polution_data = pd.read_csv("data/pollution_us_2000_2016.csv")

In [45]:
polution_data.head(5)

,Unnamed: 0,State Code,County Code,Site Num,Address,State,County,City,Date Local,NO2 Units,...,SO2 Units,SO2 Mean,SO2 1st Max Value,SO2 1st Max Hour,SO2 AQI,CO Units,CO Mean,CO 1st Max Value,CO 1st Max Hour,CO AQI
0,0,4,13,3002,1645 E ROOSEVELT ST-CENTRAL PHOENIX STN,Arizona,Maricopa,Phoenix,2000-01-01,Parts per billion,...,Parts per billion,3.000000,9.0,21,13.0,Parts per million,1.145833,4.2,21,NaN
1,1,4,13,3002,1645 E ROOSEVELT ST-CENTRAL PHOENIX STN,Arizona,Maricopa,Phoenix,2000-01-01,Parts per billion,...,Parts per billion,3.000000,9.0,21,13.0,Parts per million,0.878947,2.2,23,25.0
2,2,4,13,3002,1645 E ROOSEVELT ST-CENTRAL PHOENIX STN,Arizona,Maricopa,Phoenix,2000-01-01,Parts per billion,...,Parts per billion,2.975000,6.6,23,NaN,Parts per million,1.145833,4.2,21,NaN
3,3,4,13,3002,1645 E ROOSEVELT ST-CENTRAL PHOENIX STN,Arizona,Maricopa,Phoenix,2000-01-01,Parts per billion,...,Parts per billion,2.975000,6.6,23,NaN,Parts per million,0.878947,2.2,23,25.0
4,4,4,13,3002,1645 E ROOSEVELT ST-CENTRAL PHOENIX STN,Arizona,Maricopa,Phoenix,2000-01-02,Parts per billion,...,Parts per billion,1.958333,3.0,22,4.0,Parts per million,0.850000,1.6,23,NaN


In [46]:
polution_data.shape

(1746661, 29)

In [47]:
polution_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1746661 entries, 0 to 1746660
Data columns (total 29 columns):
 #   Column             Dtype  
---  ------             -----  
 0   Unnamed: 0         int64  
 1   State Code         int64  
 2   County Code        int64  
 3   Site Num           int64  
 4   Address            str    
 5   State              str    
 6   County             str    
 7   City               str    
 8   Date Local         str    
 9   NO2 Units          str    
 10  NO2 Mean           float64
 11  NO2 1st Max Value  float64
 12  NO2 1st Max Hour   int64  
 13  NO2 AQI            int64  
 14  O3 Units           str    
 15  O3 Mean            float64
 16  O3 1st Max Value   float64
 17  O3 1st Max Hour    int64  
 18  O3 AQI             int64  
 19  SO2 Units          str    
 20  SO2 Mean           float64
 21  SO2 1st Max Value  float64
 22  SO2 1st Max Hour   int64  
 23  SO2 AQI            float64
 24  CO Units           str    
 25  CO Mean            float64
 2

In [48]:
polution_data.describe().T

,count,mean,std,min,25%,50%,75%,max
Unnamed: 0,1746661.0,54714.136753,33729.076302,0.0000,25753.000000,53045.000000,80336.000000,134575.000000
State Code,1746661.0,22.309065,17.256205,1.0000,6.000000,17.000000,40.000000,80.000000
County Code,1746661.0,71.693807,79.480231,1.0000,17.000000,59.000000,97.000000,650.000000
Site Num,1746661.0,1118.214373,2003.103069,1.0000,9.000000,60.000000,1039.000000,9997.000000
NO2 Mean,1746661.0,12.821930,9.504814,-2.0000,5.750000,10.739130,17.713636,139.541667
NO2 1st Max Value,1746661.0,25.414848,15.999630,-2.0000,13.000000,24.000000,35.700000,267.000000
NO2 1st Max Hour,1746661.0,11.731023,7.877501,0.0000,5.000000,9.000000,20.000000,23.000000
NO2 AQI,1746661.0,23.898217,15.162805,0.0000,12.000000,23.000000,33.000000,132.000000
O3 Mean,1746661.0,0.026125,0.011370,0.0000,0.017875,0.025875,0.033917,0.095083
O3 1st Max Value,1746661.0,0.039203,0.015344,0.0000,0.029000,0.038000,0.048000,0.141000


Descriptive statistics were computed to better understand the distribution of the air quality variables. The results show significant variability in pollutant levels across different locations and time periods, highlighting the importance of incorporating environmental factors into the housing price analysis.

In [49]:
polution_data.isnull().sum()

Unnamed: 0                0
State Code                0
County Code               0
Site Num                  0
Address                   0
State                     0
County                    0
City                      0
Date Local                0
NO2 Units                 0
NO2 Mean                  0
NO2 1st Max Value         0
NO2 1st Max Hour          0
NO2 AQI                   0
O3 Units                  0
O3 Mean                   0
O3 1st Max Value          0
O3 1st Max Hour           0
O3 AQI                    0
SO2 Units                 0
SO2 Mean                  0
SO2 1st Max Value         0
SO2 1st Max Hour          0
SO2 AQI              872907
CO Units                  0
CO Mean                   0
CO 1st Max Value          0
CO 1st Max Hour           0
CO AQI               873323
dtype: int64

The air quality daraset was introduced to capture environmrntal factors that may influence housing prices. The dataset contains detailed measurements of major air pollutants across different locations in the United States over time. The data was loaded from a CSV file and initially explored using standard data inspection techniques such as .head(), .info() and describe(). This allowed for overview of the dataset structure, data types and statistical properties of the variables. The dataset includes information such as state, city, monitoring site identifiers and date of measurement. Additionally, it contains multiple air pollution indicators, includiing nitrogen dioxide(NO2), ozone(O2), sulfur dioxide(SO2) and carbon monoxide(CO) along with their respective mean values, maximum values and Air Quality Index(AQI) measurements. The dataset is relatively large, consisting of over 1.7 million records and 29 columns which makes it suitable for robust statistical analysis. Most pollutant-related variables are already stored in numeric format, facilitating further processing and modeling.

#### State Standartization

In [ ]:
polution_data.State = polution_data.State.apply(
    lambda x: us.states.lookup(x).abbr if us.states.lookup(x) else None
)

In [ ]:
polution_data[
    polution_data.State.str.len() != 2
]

In [ ]:
polution_data.loc[
    (polution_data.City == "Washington"),
    "State"
] = "DC"

In [ ]:
polution_data[
    polution_data.State.str.len() != 2
]

In [ ]:
polution_data = polution_data[
    ~polution_data.City.isin(["Tijuana", "Mexicali", "Rosarito"])
]

In [ ]:
polution_data[
    polution_data.State.str.len() != 2
]

As part of the processing of the air quality dataset, the stae information was standardized to ensure consistency across all datasets used in the analysis. Initially, full state names were converted into official two-letter abbreviations using the **us** Python library. Any values that could not be matched to a valid U.S. state were replaced with missing values(NaN). Following this transformation validation checks were performed to identify entries that did not conform to the expected two-character state format. This inconsistencies revealed special cases such as "Washighton, D.C." where the state value could not be automatically corrected by assigningthe appropriate abbreviation("DC"). Additionally, records corresponding to non-U.S. locations (e.g. cities such as Tijuana, Mexicali and Rosarito) were identified. Since these entries fall outside the scope of the analysis which focuses on U.S. housing markets, they were excluded from the dataset.

## 4. Data Aggregation and Integration

### 4.1 Data Aggregation

After cleaning and standardizing the datasets, the next step is to aggregate the external data at the state level in order to make it compatible with the housing dataset. Since the housing data is organized using state-level geographical information, both the crime and air quality datasets will be transformed into aggregated state-level erpresentations.

For the crime data, multiple datasets corresponding to different city population sizes will be grouped by state. Key crime indicators such as violent crime, murder, robbery, assult, burglary and theft will be aggregated using mean values providing a representative estimate of crime levels for each state. 

In [ ]:
crime_all_data = crime_all_data.rename(columns={"states" : "State"})

In [ ]:
crime_grouped = crime_all_data.groupby(["State"], as_index=False)[["violent_crime", "murder", "rape", "robbery","agrv_assault","prop_crime", "burglary", "larceny", "vehicle_theft", "total_crime", "arson"]].mean()

In [ ]:
crime_grouped

For the air quality data, pollutant measurements such as NO2, O3, SO2 and CO will be grouped by state and summary statisrtics such as mean, max and standard deviation will be calculated for each pollutant. This approach captures not only the average pollution levels but also the variability and extreme values within each state.

In [ ]:
air_cols = [
    "State",
    "NO2 Mean",
    "O3 Mean",
    "SO2 Mean",
    "CO Mean"
]

In [ ]:
air_features = polution_data[air_cols].groupby("State").agg({
    "NO2 Mean": ["mean", "max", "std"],
    "O3 Mean": ["mean", "max", "std"],
    "SO2 Mean": ["mean", "max", "std"],
    "CO Mean": ["mean", "max", "std"]
}).reset_index()

In [ ]:
air_features.columns = [
    "State",
    "NO2_mean", "NO2_max", "NO2_std",
    "O3_mean", "O3_max", "O3_std",
    "SO2_mean", "SO2_max", "SO2_std",
    "CO_mean", "CO_max", "CO_std",
]

In [ ]:
air_features

### 4.2 Data Integration

After aggregating the external datasets, the next step is to integrate them with the housing dataset to create a unified analytical framework. The is achieved by merging the datasets based on the common geographical identifier, namely the state abbreviation. 
The housing dataset already contained state-level information extracted during preprocessing while both the crime and air quality datasets had been transformed into state-level summaries. This ensured compatibility across all datasets and allowed for a straightfoward merge operation.

The merging process is performed using a left join, preserving all records from the housing dataset while enriching them with additional features from the crime and air quality data. 

In [ ]:
merged_housing_crime_data = housing_data.merge(crime_grouped, left_on=["State"], right_on=["State"], how="left")

In [ ]:
merged_housing_crime_data.head(5)

Following the merge, the resulting dtaset contained a comprehensive set of features, including property characteristics, crime statistics and environmental indicators. However, due to differenes in data availability across states and datasets, some missing values were introduced, particulary in crime and air quality variables.

In [ ]:
final_merged_data = merged_housing_crime_data.merge(air_features, on="State", how="left")

In [ ]:
final_merged_data.head(5)

In [ ]:
final_merged_data.info()

In [ ]:
final_merged_data.describe().T

In [ ]:
final_merged_data.isnull().sum()

### 4.3 Handling Missing Values

Following the data integration step, a detailed analysis of missing values was performed to asses data completeness across all features. The results showed that while the core housing variables contained no missing values, several features from the crime and air quality datasets exhibited incomplete coverage.

In particular, air quality variables such as NO2, SO2, O3 and CO contained a significant number of missing values, as these measurements were not available for all states. Similarly, certain crime-related features including total crime and arson also had partial data availability.

In [ ]:
final_merged_data[final_merged_data.NO2_mean.isna()]

To ensure consistency and reliability in the modeling process, rows with missing values in key environmental indicators were removed. This step ensured that all remaining observations contained complete air quality information.

In [ ]:
final_merged_data = final_merged_data.dropna(subset="NO2_mean")

In [ ]:
final_merged_data.isnull().sum()

Additional columns with high proportion of missing values such as total_crime and arson, are removed from the feature dataset to avoid introducing noise or bias into the model.

**Address** is removed from the feature set before modeling. Although address may contain location-related information, it is excluded because each address is highly specific and behaives more like unique identifier than a general predictive feature. Including it as a categorical variable could lead to overfitting where the model memorizes individual properties rather than learning broader patterns. it can introduce noise rather than meaningful signal. Location effects are already partially captured through State, population, crime and income variables.

In [ ]:
columns_to_drop = ["total_crime", "arson", "Address"]

In [ ]:
final_merged_data = final_merged_data.drop(columns=columns_to_drop)

## 5. Exploratory Data Analysis (EDA)

### 5.1 Distribution Analysis

To better understand the structure of the data, the dsitribution of key variables such as houising prices, violent crime rates and NO2 levels was examined uing histograms.

In [ ]:
plt.figure()
plt.hist(final_merged_data.Price, bins = 50)
plt.title("Price Distribution")
plt.xlabel("Price")
plt.ylabel("Frequency")
plt.show()

The distribution of housing prices appears approximately normal with slight right skew. Most properties are concentrated around the mid-range values (approximately between 800 000 and 1 500 000) while fewer observations exist at the lower and higher extremes. This indicates a relatively balanced housing market with some high-value outliers.

In [ ]:
plt.figure()
sns.boxplot(x=final_merged_data.Price)
plt.title("Price Boxplot")
plt.show()

To further examine the destribution of housing prices and identify potential outliers, a boxplot is constructed. 

The boxplot reveals that the mojority of property prices are concentrated within a relatively narrow range around the median which appears to be approximately between 1.0 and 1.3 milion. The interquartile range(IQR) indicates that most values fall within this central range, suggesting a moderate spread in housing prices. However, several outliers are visible on both the lower and upper ends of the distribution. On the lower end, there are properties with sigificantly lower prices, while on the upper end , a number of high-value properties exceed 2 million. These outliers indicate the presence of both unusually affordable and luxury properties in the dataset. The destribution appears slightly right=skewed as there are more extreme values on the higher end of the price spectrum. This suggests that while most properties are moderately priced, a smaller number of expensive homes increase the overall range.

In [ ]:
plt.figure()
plt.hist(final_merged_data.violent_crime, bins = 50)
plt.title("Violent Crime Distribution")
plt.xlabel("Violent Crime")
plt.ylabel("Frequency")
plt.show()

In contrast, the distribution of violent crime is more uneven and right-skewed. The majority of observations are clustered within lower crime levels while a small number of states exhibit significantly higher crime rates. This extreme values suggest the presence of outliers and regional disparities in safety.

In [ ]:
plt.figure()
sns.boxplot(x=final_merged_data.violent_crime)
plt.title("Crime Boxplot")
plt.show()

To analyze the distribution of violent crime across different areas, a boxplot is created. The boxplot shows that most crime values are concentrated within a moderate range, roughly between 300 and 600 incidents. The median appears arouond the middle of this interval, indicating that typical crime levels fall within this range for the majority of observations. However, the distribution is noticeably right-skewed with several high-value outliers extending beyond 1000 and even reaching above 1500. These extreme values suggest that certain areas experience significantly higher crime rates compared to the rest of the dataset. The presence of these outliers highlights regional disparities in crime levels and indicates that crime is not uniformly distributed across locations. While most areas maintain moderate levels, a smaller number of regions contribute dispropotionately to higher crime statistics. 

In [ ]:
plt.figure()
plt.hist(final_merged_data.NO2_mean, bins = 50)
plt.title("NO2 Mean Distribution")
plt.xlabel("NO2 Mean")
plt.ylabel("Frequency")
plt.show()

The distribution of NO2 levels shows moderate variability with most values concentrated within a specific range but with some dispersion cross higher values. This indicates differences in air quality across states although the variation is less axtreme compared to crime data.

### 5.2 Correlation Analysis

To better understand the relationships between variables, a correlation matrix is generated and visualized using a heatmap. This approach allows for a clear identification of both positive and negative relationships across all numerical features in the dataset.

The heatmap reveals several distinct patterns. Strong positive correlations can be observed within groups of related variables. For example, crime-related indicators such as violent crime, robbery, burglary and assult show high positive correlations with each other. This suggests that areas with high levels of one type of crime are likely to experience higher levels of other crime types as well.

Similarly, air quality variables(NO2, O3, SO2 and CO) particularly their mean, maximum and standard deviation values also exhibit strong internal correlations. This indicates consistency in pollution measurements where higher average levels are often associated with higher peaks and variability.

When focusing on housing prices the relationships appear more moderate. Price shows a positive correlation with socio-economic variables such as average area income and population, confirming that wealthier and more populated areas tend to have higher property values. On the other hand, crime-related variables demonstrate a weak to moderate negative relationship with price, suggesting that higher crime rates may reduce housing attractiveness and value.

Air quality indicators show relatively weaker correlations with housing orices compared to economic and crime factors. However, their presence still indicates that environmental conditions may play a secondary role in influencing real estate markets.

Overall, the correlation analysis highlights clear groupings among variables and supports the inclusion of crime and environmental features in further modeling. 

In [ ]:
corr = final_merged_data.corr(numeric_only=True)
plt.figure(figsize=(10,8))
sns.heatmap(corr, cmap="coolwarm")
plt.title("Correlation Matrix")
plt.show()

### 5.3 Relationship Between Crime and Housing Prices

To further investigate the relationship between crime levels and housing prices a scatter plow is created comparing violent crime and property prices. 

The visualization shows a wide dispersion of data points, indicating that the relationship between violent crimes and housing prices is not strongly linear. While there is a slight tendency for higher crime rates to be associated with lower housing prices, the pattern is not clearly defined. Most observations are concentrated in the range of moderate crime levels (approximately 200-600) where housing prices vary significantly. This suggests that other factors, such as income, population and housing characteristics may have a stronger influence on price than crime alone.

Additionally, some outliers are visible at higher crime levels but they do not follow a consistent trend in relation to price. This further supports the conclusion that violent crime while relevant is not a dominant predictor of housing prices when considered independently.

In [ ]:
plt.figure()
plt.scatter(final_merged_data.violent_crime, final_merged_data.Price, s = 1)
plt.xlabel("Violent Crime")
plt.ylabel("Price")
plt.title("Crime vs Price")
plt.show()

### 5.4 Relationship Between Income and Housing Prices

To better understand the key drivers of housing prices, a scatter plow is created comparing average area income and property prices. The visualization reveals a clear and strong positive relationship between income and housing prices. As the average income in an area increases, housing prices tend to rise accordingly. The data points form an upward-slopping pattern indicating a relatively strong linear correlation.

Most observations are clustered inn the mid-income range (approximately 50 000 - 80 000) where prices also show a consistent increase. Higher-income areas(above 80 000) are generally associated with higher property prices, oftern exceeding 1.5 milion. Although there is some spread in the data, the overall trend remains clear, suggesting that income is one of the most significant predictors of housing prices in the dataset. Compared to variables such as crime and air quality, income demonstrates a much stronger and more consistent relationship with price. This finding aligns with economic expectations as higher-income populations typically have greater purchasing power which drives demand and increases property values.

In [ ]:
plt.figure()
plt.scatter(final_merged_data["Avg. Area Income"], final_merged_data.Price, s = 1)
plt.xlabel("Avg. Area Income")
plt.ylabel("Price")
plt.title("Avg. Area Income vs Price")
plt.show()

### 5.5 Relationship Between Air Quality and Housing Prices

To examine the potential impact of air quality on housing prices, a scatterbplot is created comparing NO2 mean leveks and property prices. The visualization shows a widely scattered distribution of data points with no clear linear pattern between NO2 levels and housing prices. Property values appear to vary significantly across all levels of air pollution indicating that NO2 concentration alone does not strongly determine price. Most observations are concentrated within a moderate NO2 range (approximately 5-15) where housing prices span almost the entire price spectrum. This suggests that both low and high- priced properties can be found in areas with similar air quality levels. Unlike imcome, which demonstrated a strong positive relationship with price, air quality exhibits only a weak and inconsistent association. Any potential relationship appears to be subtle and likely influenced by other underlying factors such as location, economic conditions and housing characteristics.

In [ ]:
plt.figure()
plt.scatter(final_merged_data.NO2_mean, final_merged_data.Price, s = 1)
plt.xlabel("NO2 Mean")
plt.ylabel("Price")
plt.title("NO2 Mean vs Price")
plt.show()

### 5.6 Trend Analysis: Crime and Air Quality vs Housing Prices

To better understand the uderlying relationships between variables, regression trend lines are added to the scatter plots using linear regression.

**Crime vs Price(Trend)**

The regression line in the *Crime vs Price* plot appears almost flat, indicating a very weak negative relationship between violent crime and housing prices. Despite the wide spread of data points, there is no strong downward trend, suggesting that increases in crime levels di not significantly reduce housing prices in a consistent way. The latge dispersion around the regression line further confirms that crime alone is not a reliable predictor of price. While some lower prices are observed at higher crime levels, the overall effect remains minimal.


In [ ]:
sns.regplot(x="violent_crime", y="Price", data=final_merged_data)
plt.title("Crime vs Price(Trend)")
plt.show()

**Air Quality vs Price(Trend)**

Similarly, the regression line in the *Air Quality vs Price* plot is nearly horizontal , indicating little to no linear relationship between NO2 levels and housing prices. The data points are widely scattered across all pollution levels and no clear upward or downward trend is visible. This suggests that air quality at least as measured by NO2 concentration has a limited direct influence on housing prices when analyzed independently.

In [ ]:
sns.regplot(x="NO2_mean", y="Price", data=final_merged_data)
plt.title("Air Quality vs Price(Trend)")
plt.show()

**Overall interpretation**

The regression trend analysis reinforces previous findings from the scatter plots and correlation matrix:
 - **Income** is strong predictor of housing prices
 - **Crime** shows a weak negative or negligible relationship
 - **Air quality** has little to no direct impact on prices

These results indicate that socio-economic factors play a more significant role in determining housing prices compared to environmental or crime-related variables.

### 5.7 Feature Importance: Correlation with Housing Prices

To identify the most influential factors affecting housing prices, a correlation analysis is performed between the target variable (Price) and all numerical features. The results are visualized using a horizontal bar char, sorted by correlation strength. The analysis shows that average area income has the strongest positive correlation with housing prices making it the most significant predictor iin the dataset. This confirms previous findings from the scatter plot analysis where income demonstrated a clear upward trend with the price. 

Other variables such as average house age, area population and number of rooms also show moderate positive correlations indicating that structural and demographic characteristics contribute to the price variation although to lesser extent than income. 

In contrast crime-related variables (e.g. violent crime, roberry, burglary) exhibit very weak or near-zero correlations with price. This suggests that crime levels do not have a strng direct influence on housing prices in this dataset.

Similarly, air quality indicators (NO2, SO2, CO) show minimlal correlation with price reinforcing earlier observations that environmental factors have a limited standalone impact.

In [ ]:
corr_price = final_merged_data.corr(numeric_only=True)["Price"].sort_values()

plt.figure()
corr_price.plot(kind="barh")
plt.title("Correlation with Price")
plt.show()

## 6. Modeling

### 6.1. Model Selection

In order to predict housing prices a **Linear Regression** model is selected as a baseline approach. This model is appropriate for understanding the linear relationship between the independent variables(features)  and the target variable(Price).

After handling missing values and ensuring data consistency, the next step involved selecting the most relevant features for the predictive modeling process. During this stage, variables that were either redudant or contained a high proportioin of missing values were removed from the dataset. In particular, features such as total_crime and arson were excluded, as they exhibited incomplete coverage and could negatively impact model performance.Additionally, the target variable (Price) will be separated from the feature set to prepare the data for supervised learning. The dataset will be divided into input features(X) and target variable(y), where X includes housing characteristics, crime statistics and air quality indicators while y represents the housing price. This separation is essential for training machine learning models allowing them to learn the relationship between the predictors and the target variables. </br>
A final validation step will be performed to confirm that the feature matrix contained no missing values. This ensures that the dataset is fully prepared for modeling and eliminated the need for additional imputation techniques. The resulting feature set combines socio-economic, environmental and safety-related variables providing a comprehensive representation of the factors that may influence housing prices.

In [ ]:
columns_to_drop = "Price"

In [ ]:
X = final_merged_data.drop(columns=columns_to_drop)
y = final_merged_data.Price

In [ ]:
X.isnull().sum()

In [ ]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category", "string"]).columns.tolist()

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="mean")), 
    ("scaler", StandardScaler())
])

In [ ]:
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")), 
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

In [ ]:
preprocessor = ColumnTransformer(transformers=[("num", numeric_transformer, numeric_features), ("cat", categorical_transformer, categorical_features)])

In [ ]:
preprocessor

### 6.2 Train-Test Split

The dataset is split into training and testing sets to aveluate the model's performance on unseen data. A standard 80/20 split is used.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=42)

### 6.3 Model Training

In [ ]:
# Pipeline with Linear Regression
linear_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])


In [ ]:
linear_model

In [ ]:
linear_model.fit(X_train, y_train)

### 6.4 Predictions

In [ ]:
y_prediction = linear_model.predict(X_test)

In [ ]:
y_prediction

### 6.5 Model Evaluation

To evaluate the performance of the model, the following metric are used:
 - **MAE(Mean Absolute Error)** - average absolute difference between predicted and actual values
$$MAE = (1/n) Σ |yi - ŷi|$$

where yi is the actual value and ŷi is the predicted value. It provides an easily interpretable measure of prediction error in the same units as the target variable. Lower MAE values indicate better model performance.

In [ ]:
mean_abs_error = mean_absolute_error(y_test, y_prediction)

In [ ]:
print('MAE:', mean_abs_error)

 - **MSE(Mean Squared Error)** - measures the average squared difference between actual and predicted values:
$$ MSE = (1/n) Σ (yi - ŷi)^2 $$

Because errors are squared, larger prediction errors receive greater penalty. This makes MSE especially useful when large deviations are undesirable.

In [ ]:
mean_sqrd_error = mean_squared_error(y_test, y_prediction)

In [ ]:
print('MSE:', mean_sqrd_error)

 - **RMSE(Root Mean Squared Error)** -is the square root of MSE:
$$RMSE = √MSE$$

The root mean square error (RMSE) measures the average difference between a statistical model’s predicted values and the actual values. Mathematically, it is the standard deviation of the residuals. Residuals represent the distance between the regression line and the data points.

RMSE quantifies how dispersed these residuals are, revealing how tightly the observed data clusters around the predicted values.

In [ ]:
rmse = np.sqrt(mean_sqrd_error)

In [ ]:
print('RMSE:', rmse)

 - **R2 Score** - measures the proportion of variance in the target variable explained by the model:

$$R^2 = 1 - (SSres / SStot)$$

where:
- SSres = residual sum of squares  
- SStot = total sum of squares

Values closer to 1 indicate stronger explanatory power and better model fit.

In [ ]:
r2score = r2_score(y_test, y_prediction)

In [ ]:
print('R2 score:', r2score)

In [ ]:
final_merged_data.corr(numeric_only=True)["Price"].sort_values(ascending=False)

### 6.6 Interpretation of Results

The Linear Regression model demonstrates strong predictive performance in estimating housing price.

 - The **Mean Absolute Error(MAE)** is approximately 79 525, meaning that on average the predicted housing prices differ from actual prices by about \$79,525. Considering the scale of house prices in the dataset, this represents a relatively moderate prediction error.
 - The **Mean Squared Error(MSE)** is approximately 9.9 billion reflecting the presence of some larger prediction errors as this metric penalizes larger deviations more heavily.
 - The **Root Mean Squared Error(RMSE)** is approximately 99 528 meaning the model`s typical prediction error is around 100,000 USD.  Given that many properties are valued around \$1 million or more, this indicates reasonable predictive accuracy.
 - The model achieved an **R2 score** of 0.917 meaning that 91.7% of the variation in housing prices is explained by the model. This suggests an excellent model fit.

 **Key Findings**
 The results support the findings from the exploratory  analysis:
  - **Average Area Income** is one of the strongest predictors of house prices.
  - Housing characteristics such as **number of rooms, bedrooms and area population** also contribute significantly.
  - **Crime rates** and **air quality indicators** show weaker influence on price prediction. 

### 6.7 Additional Model Validation

To further evaluate the drivers of model performance, two additional robustness tests were conducted.

#### Model without Avg. Area Income
When the strongest predictor, Avg. Area Income, was removed, the model’s R2 score dropped from 0.917 to 0.492. This substantial decline confirms that income plays a central role in explaining housing prices and contributes significantly to the model’s predictive power.

In [ ]:
X2 = X.drop(columns=["Avg. Area Income"])

def build_preprocessor(X):
    numeric_features = X.select_dtypes(
        include=["int64","float64"]
    ).columns.tolist()

    categorical_features = X.select_dtypes(
        include=["object","category","string"]
    ).columns.tolist()

    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ])

    return preprocessor

def build_model(X):
    preprocessor = build_preprocessor(X)

    model = Pipeline([
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ])

    return model

def evaluate_model(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true,y_pred),
        "MSE": mean_squared_error(y_true,y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true,y_pred)),
        "R2": r2_score(y_true,y_pred)
    }

def train_model(X, y, test_size=0.2, random_state=42):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = test_size, random_state=random_state)
    model = build_model(X)
    model.fit(X_train, y_train)
    y_prediction = model.predict(X_test)

    results = evaluate_model(y_test, y_prediction)

    return model, results


model_without_inc, full_results_without_inc = train_model(X2,y)

In [ ]:
model_without_inc

In [ ]:
full_results_without_inc

#### Crime and Air Quality Only Model
A separate model using only crime and pollution variables produced an R2 score of approximately -0.004. This indicates these variables alone have almost no predictive power for housing prices and perform slightly worse than a baseline mean prediction. These tests reinforce that socioeconomic variables dominate housing price prediction, while crime and air quality have limited standalone influence.

In [ ]:
features = ["violent_crime", "burglary", "robbery", "NO2_mean", "O3_mean", "SO2_mean", "CO_mean"]
X_env = final_merged_data[features]


def build_preprocessor(X):
    numeric_features = X.select_dtypes(
        include=["int64","float64"]
    ).columns.tolist()

    categorical_features = X.select_dtypes(
        include=["object","category","string"]
    ).columns.tolist()

    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ])

    return preprocessor

def build_model(X):
    preprocessor = build_preprocessor(X)

    model = Pipeline([
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ])

    return model

def evaluate_model(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true,y_pred),
        "MSE": mean_squared_error(y_true,y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true,y_pred)),
        "R2": r2_score(y_true,y_pred)
    }

def train_model(X, y, test_size=0.2, random_state=42):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = test_size, random_state=random_state)
    model = build_model(X)
    model.fit(X_train, y_train)
    y_prediction = model.predict(X_test)

    results = evaluate_model(y_test, y_prediction)

    return model, results


model_env, full_results_env = train_model(X_env,y)

In [ ]:
model_env

In [ ]:
full_results_env

### Comparative Results Interpretation

Three model specifications were compared:

| Model | R2 Score |
|------|---------|
| Full Model | 0.917 |
| Without Income | 0.492 |
| Crime + Air Only | -0.004 |

 - The comparison highlights the dominant role of socioeconomic factors, especially income, in predicting housing prices. 
 - Removing income reduced explanatory power by nearly half, showing it is a critical driver in the model.
 - The crime and air-quality-only model showed virtually no predictive capability, suggesting these factors have weak direct effects on housing prices when considered independently.

## 7. Conclusion

This project examined how socioeconomic, crime, and air quality factors relate to housing prices in the United States using linear regression.The full model achieved strong predictive performance, with an R2 score of 0.917, indicating that the model explains approximately 91.7% of the variance in housing prices.

Additional validation tests provided deeper insight into what drives this performance. Removing Avg. Area Income reduced the R2 score substantially to 0.492, confirming income as one of the strongest determinants of housing prices.

A separate model using only crime and air quality variables produced almost no predictive power (R2 ≈ -0.004), suggesting these factors have limited standalone influence on housing values in this dataset.

Overall, the findings indicate that:
- Socioeconomic variables are the primary drivers of housing prices.
- Income is the dominant predictor in the model.
- Crime and pollution indicators contribute only marginally when isolated.
- Housing prices appear to be driven far more by economic fundamentals than environmental or crime-related factors.

The project demonstrates both strong predictive modeling performance and the importance of testing model robustness to better interpret feature contributions.

#### References
- Scikit-learn Documentation: </br>
  https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_absolute_error.html </br>
  https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html </br>
  https://scikit-learn.org/stable/modules/generated/sklearn.metrics.r2_score.html

https://statisticsbyjim.com/regression/root-mean-square-error-rmse/
